# anywidget

## Example 1: a counter widget defined in Python

In [ ]:
# Example 1: an AnyWidget subclass with an inline ES module and a synced traitlet
import anywidget
import ipywidgets
import traitlets

class Counter(anywidget.AnyWidget):
    _esm = "function render({ model, el }) { el.textContent = model.get('value'); }\nexport default { render };"
    _css = ".counter { font-weight: bold; }"
    value = traitlets.Int(0).tag(sync=True)

counter = Counter(value=3)
assert isinstance(counter, ipywidgets.DOMWidget)
state = counter.get_state()
assert state["value"] == 3 and "export default" in state["_esm"] and ".counter" in state["_css"]
print("Example 1:", type(counter).__name__, {k: state[k] for k in ("value", "_model_name", "_model_module")})

## Example 2: traitlet changes reach observers and the synced state

In [ ]:
# Example 2: setting a synced traitlet fires observers and changes the state sent to the front end
import anywidget
import traitlets

class Slider(anywidget.AnyWidget):
    _esm = "export default { render({ model, el }) {} };"
    position = traitlets.Float(0.0).tag(sync=True)
    label = traitlets.Unicode("x").tag(sync=True)

seen = []
slider = Slider()
slider.observe(lambda change: seen.append((change["name"], change["new"])), names=["position", "label"])
slider.position, slider.label = 0.5, "speed"
assert seen == [("position", 0.5), ("label", "speed")], seen
assert slider.get_state(["position", "label"]) == {"position": 0.5, "label": "speed"}
print("Example 2:", seen)

## Example 3: ES module and CSS read from files

In [ ]:
# Example 3: _esm and _css given as file paths are read into the widget (FileContents)
import pathlib, tempfile
import anywidget

folder = pathlib.Path(tempfile.mkdtemp())
(folder / "widget.js").write_text("export default { render({ el }) { el.innerText = 'from a file'; } };\n")
(folder / "widget.css").write_text(".from-file { color: teal; }\n")

class FileWidget(anywidget.AnyWidget):
    _esm = folder / "widget.js"
    _css = folder / "widget.css"

state = FileWidget().get_state()
assert "from a file" in state["_esm"] and "teal" in state["_css"], state
print("Example 3:", state["_esm"].strip()[:60], "|", state["_css"].strip())

## Example 4: the experimental API on an evented dataclass

In [ ]:
# Example 4: anywidget.experimental.widget + psygnal.evented make a plain dataclass displayable and synced
import dataclasses
import psygnal
import anywidget.experimental as experimental

@experimental.widget(esm="export default { render({ model, el }) { el.textContent = model.get('x'); } };")
@psygnal.evented
@dataclasses.dataclass
class Point:
    x: int = 0
    y: int = 0

point = Point(x=1, y=2)
data, _metadata = point._repr_mimebundle_()
assert "application/vnd.jupyter.widget-view+json" in data, list(data)
view = point._repr_mimebundle_
assert view._get_state(point, None) == {"x": 1, "y": 2}
point.x = 5  # psygnal emits the change; the state the front end reads follows it
assert view._get_state(point, None)["x"] == 5
print("Example 4:", data["application/vnd.jupyter.widget-view+json"]["model_id"] == view.model_id, view._get_state(point, None))

## Example 5: plotly's FigureWidget is built on anywidget

In [ ]:
# Example 5: plotly 7's FigureWidget constructs (without anywidget it raised plotly's own ImportError)
import anywidget
import plotly.graph_objects as go

fig = go.FigureWidget(data=[go.Bar(x=["a", "b", "c"], y=[3, 1, 2])], layout={"title": {"text": "anywidget"}})
assert isinstance(fig, anywidget.AnyWidget), type(fig).__mro__
fig.data[0].y = [1, 2, 3]
assert tuple(fig.data[0].y) == (1, 2, 3) and fig.layout.title.text == "anywidget"
print("Example 5:", type(fig).__name__, [base.__name__ for base in type(fig).__mro__[1:4]])